# Prática 2 — Cache TTL de tool

A mesma pergunta é executada duas vezes sem cache e duas vezes com cache.

## Fluxo da tool com TTL

```mermaid
flowchart LR
    U[Pergunta do usuário] --> A[Agente]
    A --> T[Tool de consulta]
    T --> K{Chave existe<br/>e TTL está válido?}

    K -- Sim: cache hit --> C[(Cache TTL)]
    C --> R[Resultado da tool]

    K -- Não: cache miss --> DB[(Autonomous Database)]
    DB --> W[Grava resultado<br/>com expiração]
    W --> C

    R --> L[LLM gera resposta]
    L --> U

    classDef cache fill:#fff4cc,stroke:#b8860b,stroke-width:2px;
    class C,K,W cache;
```

Enquanto o TTL estiver válido, chamadas repetidas reutilizam o resultado da tool e não executam novamente a query no banco.

In [1]:
# 0. Instalação reproduzível (roteamento e memória não são necessários aqui)
import subprocess, sys
if subprocess.run([sys.executable, "-m", "pip", "--version"], capture_output=True).returncode:
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--only-binary=:all:",
    "python-dotenv==1.2.4", "openai==3.57.0", "oracledb==4.0.2", "pandas==3.1.1"
])
print("Bibliotecas instaladas. Se foi a primeira execução, reinicie o kernel.")

Bibliotecas instaladas. Se foi a primeira execução, reinicie o kernel.


In [2]:
# 1. Carrega o .env e prepara o wallet mTLS sem expor segredos
from pathlib import Path
import os, tempfile, zipfile
from dotenv import load_dotenv

ROOT = Path.cwd() if (Path.cwd() / ".env").is_file() else Path.cwd().parent
load_dotenv(ROOT / ".env", override=True)
required = ["OCI_GENAI_API_KEY", "OCI_GENAI_BASE_URL", "OCI_GENAI_MODEL",
            "DB_USER", "DB_PASSWORD", "DB_DSN"]
missing = [x for x in required if not os.getenv(x)]
if missing: raise ValueError("Preencha no .env: " + ", ".join(missing))

wallet_temp = None
wallet_dir = Path(os.getenv("DB_WALLET_DIR", ""))
wallet_ready = (
    wallet_dir.is_dir()
    and (wallet_dir / "tnsnames.ora").is_file()
    and (wallet_dir / "ewallet.pem").is_file()
)
if os.getenv("DB_WALLET_ZIP") and not wallet_ready:
    wallet_temp = tempfile.TemporaryDirectory(prefix="cafe-ia-wallet-")
    with zipfile.ZipFile(Path(os.environ["DB_WALLET_ZIP"]).expanduser()) as archive:
        for filename in ("tnsnames.ora", "ewallet.pem"):
            item = next(x for x in archive.infolist() if Path(x.filename).name == filename)
            (Path(wallet_temp.name) / filename).write_bytes(archive.read(item))
    os.environ["DB_WALLET_DIR"] = wallet_temp.name
    os.environ.setdefault("DB_WALLET_PASSWORD", os.environ["DB_PASSWORD"])
print(".env carregado; credenciais não serão exibidas.")

.env carregado; credenciais não serão exibidas.


## 2. Conexão com o Autonomous Database

In [3]:
import re, oracledb

def db_connect_kwargs():
    dsn = os.environ["DB_DSN"]
    kwargs = {"user": os.environ["DB_USER"], "password": os.environ["DB_PASSWORD"], "dsn": dsn}
    wallet = Path(os.getenv("DB_WALLET_DIR", "/etc/cafe-ia/wallet"))
    if os.getenv("DB_WALLET_ZIP") and not (
        (wallet / "tnsnames.ora").is_file() and (wallet / "ewallet.pem").is_file()
    ):
        raise RuntimeError("Wallet temporário ausente. Execute novamente a célula 1 do .env.")
    if (wallet / "tnsnames.ora").is_file() and (wallet / "ewallet.pem").is_file():
        service = re.search(r"\(SERVICE_NAME\s*=\s*([^\)]+)\)", dsn, re.I).group(1).strip().lower()
        text = (wallet / "tnsnames.ora").read_text(encoding="utf-8")
        aliases = list(re.finditer(r"^\s*([A-Za-z0-9_.-]+)\s*=", text, re.M))
        alias = next(m.group(1) for i, m in enumerate(aliases)
                     if service in text[m.start():(aliases[i+1].start() if i+1 < len(aliases) else len(text))].lower())
        kwargs.update(dsn=os.getenv("DB_WALLET_DSN", alias), config_dir=str(wallet),
                      wallet_location=str(wallet),
                      wallet_password=os.getenv("DB_WALLET_PASSWORD", os.environ["DB_PASSWORD"]))
    return kwargs

def connect():
    return oracledb.connect(**db_connect_kwargs())

with connect() as connection:
    connection.ping()
print("Banco conectado.")

Banco conectado.


## 3. Dados exclusivos da demonstração

In [4]:
# Cria duas tabelas dedicadas e repõe somente o cliente da demo.
CUSTOMERS = "CAFE_IA_DEMO_CUSTOMERS"
EVENTS = "CAFE_IA_DEMO_EVENTS"

with connect() as connection:
    with connection.cursor() as cursor:
        for ddl in [
            f"""CREATE TABLE {CUSTOMERS} (
                customer_id VARCHAR2(30) PRIMARY KEY, company_name VARCHAR2(100) NOT NULL,
                plan_name VARCHAR2(40) NOT NULL, status VARCHAR2(20) NOT NULL,
                monthly_limit NUMBER NOT NULL)""",
            f"""CREATE TABLE {EVENTS} (
                event_id NUMBER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
                customer_id VARCHAR2(30) NOT NULL, event_time TIMESTAMP NOT NULL,
                request_count NUMBER NOT NULL, error_count NUMBER NOT NULL,
                latency_ms NUMBER NOT NULL)""",
        ]:
            try: cursor.execute(ddl)
            except oracledb.DatabaseError as error:
                if error.args[0].code != 955: raise

        # Se a tabela veio de uma versão anterior da demo, acrescenta apenas
        # as colunas ausentes. Não apaga nem recria a tabela.
        def ensure_columns(table, definitions):
            cursor.execute(
                "SELECT column_name FROM user_tab_columns WHERE table_name=:table_name",
                table_name=table.upper(),
            )
            existing = {row[0].upper() for row in cursor.fetchall()}
            for column, datatype in definitions.items():
                if column.upper() not in existing:
                    cursor.execute(f"ALTER TABLE {table} ADD ({column} {datatype})")

        ensure_columns(CUSTOMERS, {
            "customer_id": "VARCHAR2(30)", "company_name": "VARCHAR2(100)",
            "plan_name": "VARCHAR2(40)", "status": "VARCHAR2(20)",
            "monthly_limit": "NUMBER",
        })
        ensure_columns(EVENTS, {
            "event_id": "NUMBER",
            "customer_id": "VARCHAR2(30)", "event_time": "TIMESTAMP",
            "request_count": "NUMBER", "error_count": "NUMBER",
            "latency_ms": "NUMBER",
        })

        cursor.execute(f"""MERGE INTO {CUSTOMERS} c
            USING (SELECT 'XPTO-4821' customer_id FROM dual) d
            ON (c.customer_id=d.customer_id)
            WHEN MATCHED THEN UPDATE SET company_name='Aurora', plan_name='Enterprise',
                status='ATIVO', monthly_limit=18500
            WHEN NOT MATCHED THEN INSERT
                (customer_id, company_name, plan_name, status, monthly_limit)
                VALUES ('XPTO-4821','Aurora','Enterprise','ATIVO',18500)""")
        cursor.execute(f"DELETE FROM {EVENTS} WHERE customer_id='XPTO-4821'")
        cursor.executemany(
            f"INSERT INTO {EVENTS} "
            "(event_id,customer_id,event_time,request_count,error_count,latency_ms) "
            "VALUES (:1,:2,SYSTIMESTAMP-NUMTODSINTERVAL(:3,'HOUR'),:4,:5,:6)",
            [(4821001, "XPTO-4821", 1, 1250, 38, 920),
             (4821002, "XPTO-4821", 3, 1180, 27, 810),
             (4821003, "XPTO-4821", 7, 1320, 42, 1050),
             (4821004, "XPTO-4821", 12, 1100, 19, 740),
             (4821005, "XPTO-4821", 30, 980, 8, 430)],
        )
    connection.commit()
print("Dados da demo prontos.")

Dados da demo prontos.


## 4. Tool SQL — consulta com CTE e função analítica

In [5]:
QUERY = f"""
WITH recent_events AS (
    SELECT customer_id, request_count, error_count, latency_ms, event_time,
           ROUND(error_count / NULLIF(request_count, 0) * 100, 2) error_rate_pct,
           ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY event_time DESC) AS rn
    FROM {EVENTS}
    WHERE event_time >= SYSTIMESTAMP - INTERVAL '2' DAY
), summary AS (
    SELECT customer_id, COUNT(*) event_count,
           SUM(request_count) request_count, SUM(error_count) error_count,
           ROUND(AVG(error_rate_pct), 2) avg_error_rate_pct,
           ROUND(AVG(latency_ms), 2) avg_latency_ms,
           MAX(CASE WHEN rn=1 THEN latency_ms END) latest_latency_ms
    FROM recent_events GROUP BY customer_id
)
SELECT c.customer_id, c.company_name, c.plan_name, c.status, c.monthly_limit,
       NVL(s.event_count,0) event_count, NVL(s.request_count,0) request_count,
       NVL(s.error_count,0) error_count,
       NVL(s.avg_error_rate_pct,0) avg_error_rate_pct,
       NVL(s.avg_latency_ms,0) avg_latency_ms,
       NVL(s.latest_latency_ms,0) latest_latency_ms
FROM {CUSTOMERS} c LEFT JOIN summary s ON s.customer_id=c.customer_id
WHERE c.customer_id=:customer_id
"""

db_calls = 0
def customer_tool(customer_id):
    global db_calls
    db_calls += 1
    with connect() as connection, connection.cursor() as cursor:
        cursor.execute(QUERY, customer_id=customer_id)
        row = cursor.fetchone()
        columns = [x[0].lower() for x in cursor.description]
    return dict(zip(columns, row))

customer_tool("XPTO-4821")

{'customer_id': 'XPTO-4821',
 'company_name': 'Aurora',
 'plan_name': 'Enterprise',
 'status': 'ATIVO',
 'monthly_limit': 18500,
 'event_count': 5,
 'request_count': 5830,
 'error_count': 134,
 'avg_error_rate_pct': 2.21,
 'avg_latency_ms': 790,
 'latest_latency_ms': 920}

## 5. Agente: chama a tool e entrega o resultado ao mesmo LLM

In [6]:
from time import perf_counter, monotonic
from openai import OpenAI

client = OpenAI(api_key=os.environ["OCI_GENAI_API_KEY"],
                base_url=os.environ["OCI_GENAI_BASE_URL"], timeout=90)
MODEL = os.environ["OCI_GENAI_MODEL"]
QUESTION = "Qual é a situação do cliente XPTO-4821 e o risco recente?"

def ask_agent(tool_result):
    prompt = f"Pergunta: {QUESTION}\nResultado da tool SQL: {tool_result}"
    started = perf_counter()
    response = client.responses.create(
        model=MODEL, instructions="Responda em até três frases, sem inventar dados.", input=prompt)
    return {"answer": response.output_text,
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "llm_s": perf_counter() - started}

class TTLCache:
    def __init__(self, ttl_seconds):
        self.ttl_seconds, self.items = ttl_seconds, {}
    def get_or_load(self, key, loader):
        item = self.items.get(key)
        if item and item[0] > monotonic(): return item[1], True
        value = loader(); self.items[key] = (monotonic() + self.ttl_seconds, value)
        return value, False

## 6. Duas chamadas sem cache

In [7]:
rows = []
for attempt in (1, 2):
    before = db_calls; started = perf_counter()
    data = customer_tool("XPTO-4821")
    tool_s = perf_counter() - started
    llm = ask_agent(data)
    rows.append({"estratégia": "Sem cache", "tentativa": attempt, "fonte": "banco",
                 "tool_s": tool_s, "llm_s": llm["llm_s"],
                 "tempo_total_s": tool_s + llm["llm_s"],
                 "tokens": llm["input_tokens"] + llm["output_tokens"],
                 "chamadas_banco": db_calls - before, "resposta": llm["answer"]})
rows[-1]

{'estratégia': 'Sem cache',
 'tentativa': 2,
 'fonte': 'banco',
 'tool_s': 0.19053319992963225,
 'llm_s': 2.587379299919121,
 'tempo_total_s': 2.777912499848753,
 'tokens': 333,
 'chamadas_banco': 1,
 'resposta': 'O cliente XPTO-4821 (Aurora) está **ativo** no plano Enterprise, com 5.830 solicitações frente ao limite mensal de 18.500. O risco recente é o aumento da latência para **920 ms**, acima da média de **790 ms**, além de 134 erros e taxa média de erro de **2,21%**; sem limites de referência, não é possível classificar a gravidade.'}

## 7. Duas chamadas com cache TTL

In [8]:
cache = TTLCache(ttl_seconds=300)
for attempt in (1, 2):
    before = db_calls; started = perf_counter()
    data, hit = cache.get_or_load("XPTO-4821", lambda: customer_tool("XPTO-4821"))
    tool_s = perf_counter() - started
    llm = ask_agent(data)
    rows.append({"estratégia": "Cache TTL", "tentativa": attempt,
                 "fonte": "cache" if hit else "banco", "tool_s": tool_s,
                 "llm_s": llm["llm_s"], "tempo_total_s": tool_s + llm["llm_s"],
                 "tokens": llm["input_tokens"] + llm["output_tokens"],
                 "chamadas_banco": db_calls - before, "resposta": llm["answer"]})
rows[-1]

{'estratégia': 'Cache TTL',
 'tentativa': 2,
 'fonte': 'cache',
 'tool_s': 6.00004568696022e-06,
 'llm_s': 1.9925480999518186,
 'tempo_total_s': 1.9925540999975055,
 'tokens': 277,
 'chamadas_banco': 0,
 'resposta': 'O cliente XPTO-4821 (Aurora) está **ATIVO** no plano Enterprise, com 5.830 requisições frente ao limite mensal de 18.500. O risco recente é o aumento da latência: 920 ms na medição mais recente, acima da média de 790 ms; além disso, foram registrados 134 erros, com taxa média de 2,21%.'}

## 8. Comparativo final

In [ ]:
import pandas as pd
detail = pd.DataFrame(rows)
summary = detail.groupby("estratégia", as_index=False).agg(
    chamadas_banco=("chamadas_banco", "sum"), tool_s=("tool_s", "sum"),
    tempo_total_s=("tempo_total_s", "sum"), tokens=("tokens", "sum"))
base = summary[summary["estratégia"] == "Sem cache"].iloc[0]
for column in ["chamadas_banco", "tool_s", "tempo_total_s", "tokens"]:
    summary[f"dif_{column}_vs_sem_cache_%"] = (summary[column] / base[column] - 1) * 100
display(detail.round(3)); display(summary.round(2))


,estratégia,tentativa,fonte,tool_s,llm_s,tempo_total_s,tokens,chamadas_banco,resposta
0,Sem cache,1,banco,0.182,2.815,2.998,300,1,O cliente XPTO-4821 (Aurora) está **ATIVO** no...
1,Sem cache,2,banco,0.191,2.587,2.778,333,1,O cliente XPTO-4821 (Aurora) está **ativo** no...
2,Cache TTL,1,banco,0.174,2.104,2.278,313,1,O cliente XPTO-4821 (Aurora) está **ativo** no...
3,Cache TTL,2,cache,0.000,1.993,1.993,277,0,O cliente XPTO-4821 (Aurora) está **ATIVO** no...


,estratégia,chamadas_banco,tool_s,tempo_total_s,tokens,dif_chamadas_banco_vs_sem_cache_%,dif_tool_s_vs_sem_cache_%,dif_tempo_total_s_vs_sem_cache_%,dif_tokens_vs_sem_cache_%
0,Cache TTL,1,0.17,4.27,590,-50.0,-53.26,-26.05,-6.79
1,Sem cache,2,0.37,5.78,633,0.0,0.00,0.00,0.00


Percentual negativo = redução; a resposta da LLM pode manter os mesmos tokens.
